In [1]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

drop reason:

or.takeaway.order: 'quit_order',                  # 瀏覽菜單
# or.takeaway.view-basket:'quit_basket',            # 購物籃
or.takeaway.checkout: 'quit_checkout',            # 選擇優惠及下單
or.takeaway.place-order: 'quit_place-order',      # Rice dollar + 繼續

or.takeaway.payment

In [5]:
sql = """
WITH drop_sessions AS (
  SELECT
    DeviceId,
    SessionId,
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
  GROUP BY DeviceId, SessionId
  HAVING
    COUNTIF(LOWER(TRIM(EventAction)) = 'or.takeaway.order') > 0         -- must hv order event
    AND COUNTIF(                                                -- session hv no pay event
      LOWER(TRIM(EventAction)) = 'or.takeaway.pay'
      OR LOWER(COALESCE(EventLabelRaw, '')) LIKE '%or.takeaway.pay%'
    ) = 0
),

sampled_drop_sessions AS (
  SELECT
    DeviceId,
    SessionId
  FROM drop_sessions
  ORDER BY RAND()
  LIMIT 1000
)

SELECT pv.SessionId, pv.DeviceId, pv.Time, pv.EventAction, pv.EventLabelRaw
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN sampled_drop_sessions AS ds
  ON pv.DeviceId = ds.DeviceId
  AND pv.SessionId = ds.SessionId
ORDER BY pv.DeviceId, pv.SessionId, pv.Time;
"""

df_bq = client.query(sql).result().to_dataframe()
df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,EventAction,EventLabelRaw
0,1087,0047fccf-357b-4b80-b021-fe12a4b3900b,2026-09-15 14:17:50+00:00,or.app.start,CityID:0;Lang:hk;Ver:7.20.5;device:iPhone 14 Pro
1,1087,0047fccf-357b-4b80-b021-fe12a4b3900b,2026-09-15 14:17:51+00:00,or.deeplink.get-poi,CityID:0;Lang:hk;Ver:7.20.5;tracking:u3mwFAA;P...
2,1087,0047fccf-357b-4b80-b021-fe12a4b3900b,2026-09-15 14:17:51.100000+00:00,or.poi.get-overview,CityID:0;Lang:hk;Ver:7.20.5;tracking:u3mwFAA;P...
3,1087,0047fccf-357b-4b80-b021-fe12a4b3900b,2026-09-15 14:17:51.100000+00:00,or.poi.get-details,CityID:0;Lang:hk;Ver:7.20.5;tracking:u3mwFAA;P...
4,1087,0047fccf-357b-4b80-b021-fe12a4b3900b,2026-09-15 14:18:06+00:00,or.poi.get-photos.all,CityID:0;Lang:hk;Ver:7.20.5;tracking:u3mwFAA;P...
...,...,...,...,...,...
152205,1030580,ffe43503-b0b2-45af-bec3-2133edcd1d24,2026-09-15 11:11:26.400000+00:00,or.explore.reel.photo,CityID:0;Lang:hk;Ver:7.20.5;poiId:49662;SrcPho...
152206,1030580,ffe43503-b0b2-45af-bec3-2133edcd1d24,2026-09-15 11:11:34+00:00,impression.poi,CityID:0;Lang:hk;Ver:7.20.5;POIID:11220;Type:G...
152207,1030580,ffe43503-b0b2-45af-bec3-2133edcd1d24,2026-09-15 11:11:34+00:00,impression.poi,CityID:0;Lang:hk;Ver:7.20.5;POIID:537520;Type:...
152208,1030580,ffe43503-b0b2-45af-bec3-2133edcd1d24,2026-09-15 11:11:34+00:00,impression.poi,CityID:0;Lang:hk;Ver:7.20.5;POIID:1128979;Type...


In [ ]:
#df_bq.to_csv("session.csv", index=False, encoding="utf-8-sig")


In [8]:
count_quit_point = {
    "quit_order": 0,
    "quit_checkout": 0,
    "quit_place-order": 0,
}

dict_quit_point = {
    "quit_order": 1,
    "quit_checkout": 2,
    "quit_place-order": 3,
}

In [ ]:
# 3：按時間倒序查詢session
tracking_source = df_bq.copy()
tracking_source["Time"] = pd.to_datetime(tracking_source["Time"])

tracking_source = tracking_source.sort_values(
    ["DeviceId", "SessionId", "Time"],
    kind="stable",
)
tracking_source["normalized_action"] = (
    tracking_source["EventAction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
quit_action_mapping = {
    "or.takeaway.order": "quit_order",
    "or.takeaway.checkout": "quit_checkout",
    "or.takeaway.place-order": "quit_place-order",
}


def classify_quit_point(session_events):
    actions = session_events["normalized_action"].tolist()

    # 倒序找最後一個quit point
    for action in reversed(actions):
        quit_name = quit_action_mapping.get(action)

        if quit_name is not None:
            return dict_quit_point[quit_name]

    return pd.NA

# 每個 DeviceId + SessionId 產生一個 quit_point
drop_session_summary = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], sort=False)
    .apply(classify_quit_point)
    .rename("quit_point")
    .reset_index()
)

# 將 quit_point 的數字轉回名稱後計數
quit_point_name_mapping = {
    1: "quit_order",
    2: "quit_checkout",
    3: "quit_place-order",
}

classified_quit_names = drop_session_summary["quit_point"].map(
    quit_point_name_mapping
)

count_quit_point.update(
    classified_quit_names.value_counts().to_dict()
)

#display(drop_session_summary)

{'quit_order': 985, 'quit_checkout': 12, 'quit_place-order': 3}

In [12]:
quit_point_result = (
    pd.DataFrame.from_dict(
        count_quit_point,
        orient="index",
        columns=["count"],
    )
    .rename_axis("quit_point")
    .reset_index()
)

total_drop_sessions = quit_point_result["count"].sum()

quit_point_result["%"] = (
    quit_point_result["count"]
    .div(total_drop_sessions)
    .mul(100)
    .round(2)
)

quit_point_result

,quit_point,count,%
0,quit_order,985,98.5
1,quit_checkout,12,1.2
2,quit_place-order,3,0.3
